# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution
Exploration with `mlcroissant`

This notebook demonstrates how to load and analyze the FAIRˆ² dataset package using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined by a Croissant schema at:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

All entities in the metadata (record sets, fields, columns, etc.) are referenced using their `@id`. This ensures reproducibility and fine-grained data access.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')  # Suppress pandas warnings for demo

# Define the Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print dataset high-level info
print(f"{metadata.name}: {metadata.description}\n")

### Dataset Description
Tabular dataset of 77 cancer survivors with second primary colorectal cancer, including clinical and pathological variables such as demographics, comorbidities, first and second primary cancer types, treatment history, intervals between diagnoses, anatomical location of colorectal cancer, histopathological subtype, presence of distant metastasis, and microsatellite instability status. Data supports investigation of clinicopathological predictors and distribution of MSI-H phenotype.

## 2. Data Overview
Review all available record sets, their `@id`, and fields. This step programmatically lists all record sets and their fields using their `@id` for all further referencing.


In [ ]:
# List all record sets and their fields/columns by @id

record_set_objs = list(metadata.record_sets)

print(f"Number of record sets in this dataset: {len(record_set_objs)}\n")
record_set_id_list = []

for rs in record_set_objs:
    print(f"RecordSet: {rs.name}  (@id: {rs.id})")
    record_set_id_list.append(rs.id)
    # List fields (usually references 'fields' or 'columns', not direct field attributes)
    if hasattr(rs, 'fields') and rs.fields:
        print("  Field @ids:")
        for field in rs.fields:
            print(f"    - {field.id} ({field.name})")
    elif hasattr(rs, 'columns') and rs.columns:
        print("  Columns @ids:")
        for col in rs.columns:
            print(f"    - {col.id} ({col.name})")
    print('------')

Below, we list some sample records from the **first record set** (replace with desired `@id` if needed).

In [ ]:
# Inspect first few records in the first record set (by @id)

# For demonstration, we use the first available record set @id
record_set_id = record_set_id_list[0]
for i, record in enumerate(dataset.records(record_set=record_set_id)):
    print(json.dumps(record, indent=2))
    if i >= 2:
        break

## 3. Data Extraction
Load data from all record sets into pandas DataFrames for analysis. Each is referenced by its `@id` key.


In [ ]:
# Extract all record sets as DataFrames by @id

dataframes = {}
for record_set in record_set_id_list:
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df
    print(f"Loaded DataFrame for RecordSet @id: {record_set} with shape {df.shape}")

# Show columns in the first record set
print(f"\nFields/columns in chosen RecordSet {record_set_id}:")
print(dataframes[record_set_id].columns.tolist())

dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field for demonstration and perform:
- Filtering for records above a threshold
- Normalization
- Grouping by a categorical field

**Note:** Fields are referenced using their `@id` and mapped to column names below. Replace with appropriate column names if needed.

In [ ]:
# Choose field names by inspecting columns above. Example uses likely candidates ('Age' / 'IntervalMonths' etc.)
# Replace 'age' and 'sex' below with actual column names or @id if different.

df = dataframes[record_set_id]

# Try to automatically choose a numeric column (fallback if fields are unknown):

numeric_field = None
for col in df.columns:
    if df[col].dtype in ['int64', 'float64']:
        numeric_field = col
        break

# Manually set if required (common names: 'Age', 'IntervalBetweenCancers', etc.)
if not numeric_field:
    # Try common field
    for cn in ['Age', 'IntervalMonths', 'IntervalBetweenCancers']:
        if cn in df.columns:
            numeric_field = cn
            break

print(f"Selected numeric field: {numeric_field}")

group_field = None
# Try to find a categorical field ('Sex', 'Gender', 'AnatomicalLocation', etc.)
for cn in ['Sex', 'Gender', 'AnatomicalLocation', 'MSI_High', 'Comorbidity']:
    if cn in df.columns:
        group_field = cn
        break
if not group_field:
    # Fallback: use first object-typed column
    for col in df.columns:
        if df[col].dtype == 'object':
            group_field = col
            break
print(f"Selected group field: {group_field}")

# Filter for values > threshold
if numeric_field:
    df_clean = df.copy()
    df_clean[numeric_field] = pd.to_numeric(df_clean[numeric_field], errors='coerce')
    threshold = df_clean[numeric_field].mean() if df_clean[numeric_field].mean() else 10
    filtered_df = df_clean[df_clean[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold:.1f}:")
    print(filtered_df.head())

    # Normalize selected numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group field and show group statistics
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['mean', 'count', 'std'])
        print(f"\nGrouped {numeric_field} stats by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution and group differences using histograms and box plots. Requires `matplotlib`/`seaborn`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True, color='teal', alpha=0.7)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    if group_field in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, exploring, and visualizing a tabular clinical oncology dataset using the Croissant standard and `mlcroissant`. Fields, columns, and record sets were referenced by their `@id` to ensure schema consistency.

**Key Observations:**
- The dataset contains a richly annotated, small cohort (N=77) including demographics, cancer characteristics, and outcome-relevant biomarkers.
- Using the field and record set `@id`s, you can select and explore any part of the dataset for downstream analysis.
- Further domain-specific analysis can assess clinicopathological predictors and the anatomical distribution of MSI-H phenotype in cancer survivors.